# 2 — Map D2/D4 queries onto the D10 SCANVI reference

Iterates over D2_DZ, D2_Lapa, D4_DZ, D4_Lapa. For each query: `prepare_query_anndata` → `load_query_data` → short fine-tune → predict argmax + soft probabilities, with `Ambiguous_<argmax>` flagging when `scanvi_confidence < 0.5`.

Per-query outputs:
- `data/data-objects/cellassign/<query>_scanvi_predictions.h5ad`
- `data/analysis-files/cell_assign_dfs/<query>_scanvi_probabilities.csv`

In [ ]:
import sys, gc
from pathlib import Path

_p = Path('.').resolve()
while not (_p / 'src' / 'config.py').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

from src.config import INTEGRATION_DIR, CELLASSIGN_DIR, ANALYSIS_FILES_DIR
from src.transfer_learning import map_query

ACCELERATOR = 'auto'
AMBIGUOUS_THRESHOLD = 0.5
MAX_EPOCHS_QUERY = 100

QUERY_KEYS = ['D2_DZ', 'D2_Lapa', 'D4_DZ', 'D4_Lapa']

SLIM_HVG = INTEGRATION_DIR / 'slim_hvg'
MODEL_DIR = INTEGRATION_DIR / 'scarches_model'
CSV_DIR = ANALYSIS_FILES_DIR / 'cell_assign_dfs'
CELLASSIGN_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
results = {}
for key in QUERY_KEYS:
    print(f'\n=== mapping {key} ===')
    out = map_query(
        query_slim_path=SLIM_HVG / f'{key}.h5ad',
        ref_model_dir=MODEL_DIR,
        predictions_h5ad_path=CELLASSIGN_DIR / f'{key.lower()}_scanvi_predictions.h5ad',
        predictions_csv_path=CSV_DIR / f'{key.lower()}_scanvi_probabilities.csv',
        layer='counts',
        max_epochs_query=MAX_EPOCHS_QUERY,
        ambiguous_threshold=AMBIGUOUS_THRESHOLD,
        seed=0,
        accelerator=ACCELERATOR,
    )
    results[key] = out
    gc.collect()
results

In [ ]:
# Quick post-mapping summary
import scanpy as sc
import pandas as pd

summary_rows = []
for key in QUERY_KEYS:
    a = sc.read_h5ad(results[key]['h5ad'])
    counts = a.obs['scanvi_prediction'].value_counts(normalize=True).round(3).to_dict()
    raw_counts = a.obs['scanvi_prediction_raw'].value_counts(normalize=True).round(3).to_dict()
    n_ambiguous = a.obs['scanvi_prediction'].astype(str).str.startswith('Ambiguous_').sum()
    summary_rows.append({
        'dataset': key,
        'n_cells': a.n_obs,
        'mean_confidence': float(a.obs['scanvi_confidence'].mean()),
        'pct_ambiguous': float(n_ambiguous) / a.n_obs,
        'top_predictions': counts,
    })
    del a
    gc.collect()
pd.DataFrame(summary_rows)

In [ ]:
# Confidence histograms (saved to figures/)
import matplotlib.pyplot as plt
from src.config import FIGURES_DIR
fig_dir = FIGURES_DIR / 'd2-d4-integrated' / 'sanity'
fig_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, len(QUERY_KEYS), figsize=(4*len(QUERY_KEYS), 3), sharey=True)
for ax, key in zip(axes, QUERY_KEYS):
    a = sc.read_h5ad(results[key]['h5ad'])
    ax.hist(a.obs['scanvi_confidence'], bins=40)
    ax.axvline(AMBIGUOUS_THRESHOLD, color='red', linestyle='--', linewidth=1)
    ax.set_title(key); ax.set_xlabel('confidence'); ax.set_xlim(0, 1)
    del a; gc.collect()
axes[0].set_ylabel('cells')
plt.tight_layout()
plt.savefig(fig_dir / 'confidence_histograms.pdf')
plt.show()